# Prompt Release and Experimentation

> **The story.** In the 1920s, R. A. Fisher formalized randomized experiments at Rothamsted so an observed difference could be separated from field-to-field variation. Paired comparisons sharpen that idea by holding the experimental unit fixed. Prompt releases inherit the same obligation: change one pinned component, compare like with like, and retain enough evidence to reverse the decision.
>
> **Where you are.** Riverside House already has authorized retrieval, evaluation cases, and a gateway contract. A new concise prompt improves the fixture's aggregate pass rate from 60% to 80%, but the team has not checked slices, uncertainty, assignment, or rollback evidence.
>
> **Notation.** $b_i$ is baseline pass for case $i$; $c_i$ is candidate pass; $d_i=c_i-b_i$ is the paired delta; $n_d$ is the number of discordant pairs; $p$ is an exact sign-test value; $w$ is candidate traffic share.

> **Inter-notebook contract:** [RAG evaluation](../../../genai/04-rag/05-rag-evaluation.ipynb) supplies slice-aware evaluation; [LLM evaluation](../../../genai/05-llm-evaluation/README.md) supplies regression and uncertainty discipline; the [LLM gateway](../../../genai/06-llm-gateway/06-llm-gateway.ipynb) supplies stable routing. This notebook delivers a prompt release decision and rollback evidence for the later [release registry](../04-release-registry-and-lineage/release-registry-and-lineage.ipynb).

| Step | Failure you expose | Control you build | Evidence |
|---:|---|---|---|
| 0 | A plausible aggregate improvement invites a rushed release | Frozen local fixture contract | Read-only hashes and schema checks |
| 1 | A prompt string cannot reproduce application behavior | Versioned release bundle | Component diff and canonical digest |
| 2 | Aggregate quality hides a critical regression | Slice-aware gate | Security slice blocks promotion |
| 3 | Five cases look more certain than they are | Paired comparison and uncertainty | Wins/losses/ties, exact sign test, paired interval |
| 4 | Different traffic confounds online comparison | Shadow and stable A/B assignment | Exposure-level routing checks |
| 5 | A candidate can reach users without stop evidence | Canary entry and rollback contract | Blocked canary plus retained target |
| 6 | A decision disappears into a dashboard | One auditable report | Reject candidate, retain baseline |

## 0 - The Challenge

> **The mission:** Riverside House must decide whether `prompt-riv-002` may replace accepted `prompt-riv-001` without exposing confidential material or shipping a regression hidden by an average.

**What we know so far:**

- The baseline passes 3 of 5 fixture cases.
- The candidate passes 4 of 5 fixture cases.
- **But a release is not an aggregate score, and five cases are not a population.**

**What's blocking us:** The candidate prompt asks for direct answers without caveats or uncertainty language. That change helps two incomplete baseline answers, but it may remove the refusal behavior needed when retrieved security context is incomplete.

**What this chapter unlocks:** a reproducible release object, a paired and slice-aware offline gate, an uncertainty label, stable online assignment, and evidence that either authorizes a canary or retains the rollback target.

```mermaid
flowchart LR
    A["Candidate prompt\naggregate 80%"] --> B["Failure: aggregate\nhides security loss"]
    B --> C["Paired and slice\noffline gate"]
    C --> D["Decision: critical\nslice regressed"]
    D --> E["Retain accepted\nrollback target"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Topic Space and Evidence Boundary

| Sub-topic | Coverage | Why |
|---|---|---|
| Release bundle and digest | Built and measured | Reproducibility requires all behavior-affecting pins |
| Aggregate and slice gates | Built and measured | This is the seeded Riverside failure |
| Paired comparison and exact sign test | Built and measured | Same cases make a controlled offline comparison |
| Paired bootstrap interval | Built and measured | Shows finite-sample instability without claiming significance |
| Shadow replay | Built and measured | Candidate output is observed but never served |
| Deterministic A/B assignment | Built and measured | Prevents release-dependent and request-level reassignment |
| Canary entry and rollback evidence | Built and measured | Offline failure must stop exposure |
| Sequential testing and peeking controls | Explained and illustrated | A real online test needs a pre-registered analysis plan |
| CUPED, stratification, and variance reduction | Named with a reason | Needs richer covariates than the five fixture rows |
| Multi-armed bandits | Named with a reason | Optimization under adaptive assignment is a different objective from release inference |
| Live provider or LLM evaluation | Named with a reason | Network and LLM calls are forbidden on the default path |

> **Evidence boundary:** every response and pass label is pre-recorded in the immutable Riverside fixtures. The notebook measures release mechanics, not fresh model quality.

In [ ]:
# -- Local-only setup and chapter discovery ---------------------------------
from __future__ import annotations

from collections import Counter, defaultdict
from copy import deepcopy
from hashlib import blake2b, sha256
from math import comb
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
from jsonschema import Draft202012Validator, FormatChecker

RANDOM_SEED = 20260805
random.seed(RANDOM_SEED)

def locate_chapter_dir() -> Path:
    relative = Path("learning/role-based-tracks/ai-engineer/02-prompt-release-and-experimentation")
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        for candidate_dir in (base, base / relative):
            if (candidate_dir / "prompt-release-and-experimentation.ipynb").is_file():
                return candidate_dir
    raise FileNotFoundError("Run from the repository or chapter directory.")

CHAPTER_DIR = locate_chapter_dir()
FIXTURE_DIR = CHAPTER_DIR.parent / "shared" / "prompt-release"
print(f"Chapter directory: {CHAPTER_DIR}")
print(f"Fixture directory: {FIXTURE_DIR}")
print("TAKEAWAY: the default path is local-only; no provider SDK, model, or network client is imported.")

In [ ]:
# -- Read and validate the immutable Riverside fixtures ---------------------
fixture_paths = {
    "releases": FIXTURE_DIR / "prompt-releases.json",
    "release_schema": FIXTURE_DIR / "prompt-releases.schema.json",
    "cases": FIXTURE_DIR / "paired-eval-cases.jsonl",
    "case_schema": FIXTURE_DIR / "paired-eval-case.schema.json",
    "expected": FIXTURE_DIR / "EXPECTED_OUTCOMES.md",
}
assert all(path.is_file() for path in fixture_paths.values())

shared_dir = FIXTURE_DIR.parent
fixture_version = (shared_dir / "VERSION").read_text(encoding="utf-8").strip()
fixture_manifest = json.loads((shared_dir / "fixture-manifest.json").read_text(encoding="utf-8"))
if fixture_manifest["fixture_version"] != fixture_version:
    raise RuntimeError("Fixture VERSION and fixture-manifest.json disagree")

fixture_hashes_before = {}
for name, path in fixture_paths.items():
    relative_path = path.relative_to(shared_dir).as_posix()
    expected_digest = fixture_manifest["files"].get(relative_path)
    if expected_digest is None:
        raise RuntimeError(f"Fixture manifest does not pin {relative_path}")
    actual_digest = sha256(path.read_bytes()).hexdigest()
    if actual_digest != expected_digest:
        raise RuntimeError(
            f"Stale or modified fixture: {relative_path}. "
            "Restore the pinned bytes or intentionally version the shared fixture contract."
        )
    fixture_hashes_before[name] = actual_digest

release_document = json.loads(fixture_paths["releases"].read_text(encoding="utf-8"))
release_schema = json.loads(fixture_paths["release_schema"].read_text(encoding="utf-8"))
case_schema = json.loads(fixture_paths["case_schema"].read_text(encoding="utf-8"))
paired_cases = [
    json.loads(line)
    for line in fixture_paths["cases"].read_text(encoding="utf-8").splitlines()
    if line.strip()
]

release_errors = list(Draft202012Validator(release_schema, format_checker=FormatChecker()).iter_errors(release_document))
case_validator = Draft202012Validator(case_schema, format_checker=FormatChecker())
case_errors = [error for case in paired_cases for error in case_validator.iter_errors(case)]
assert not release_errors, release_errors
assert not case_errors, case_errors
assert len(paired_cases) == 5
print(f"Verified fixture contract: {fixture_version}")
print(f"Validated {len(release_document['releases'])} releases and {len(paired_cases)} paired cases.")
print("TAKEAWAY: schema validity is the first gate, not the release decision.")

### Fixture Pitfalls and Quick Health Check

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Copy fixture rows into the notebook and edit labels | The lesson can silently make its own candidate pass |
| Right | Read shared v1 files in place and hash them | The decision remains tied to a frozen contract |
| Wrong | Join baseline and candidate by row position | Reordering changes identity |
| Right | Join by `eval_case_id` and release ID | Stable IDs survive ordering changes |

**Quick Health Check**

1. Every fixture path exists.
2. Both schemas validate.
3. Case IDs are unique.
4. Every case names the same baseline and candidate release IDs.
5. The notebook has not changed fixture bytes.

In [ ]:
# -- Run the fixture health checks ------------------------------------------
case_ids = [case["eval_case_id"] for case in paired_cases]
baseline_ids = {case["baseline_prompt_release_id"] for case in paired_cases}
candidate_ids = {case["candidate_prompt_release_id"] for case in paired_cases}
fixture_hashes_now = {name: sha256(path.read_bytes()).hexdigest() for name, path in fixture_paths.items()}

checks = {
    "paths_exist": all(path.is_file() for path in fixture_paths.values()),
    "schemas_valid": not release_errors and not case_errors,
    "case_ids_unique": len(case_ids) == len(set(case_ids)),
    "one_baseline": baseline_ids == {"prompt-riv-001"},
    "one_candidate": candidate_ids == {"prompt-riv-002"},
    "fixture_bytes_unchanged": fixture_hashes_now == fixture_hashes_before,
}
assert all(checks.values()), checks
for name, passed in checks.items():
    print(f"PASS: {name} = {passed}")
print("TAKEAWAY: the mechanism starts from stable identities and unchanged fixture bytes.")

**Reflection bridge:** Stable fixtures make the comparison reproducible, but a prompt string alone still omits four behavior-affecting pins. The release unit must widen before any score can be trusted.

---

## 1 - Version the Application Behavior, Not Just the Prompt

Riverside's question for this section: if the answer changes next week, can you reconstruct every behavior-affecting input that produced it?

A prompt template is only one component. Tool schemas change what actions are expressible, retrieval configuration changes evidence, the model alias changes generation behavior, and evaluator version changes the measured result. Promote them as one immutable release bundle.

```mermaid
flowchart LR
    P["Prompt template"] --> R["Prompt release bundle"]
    T["Tool schema version"] --> R
    I["Retrieval config"] --> R
    M["Model alias"] --> R
    E["Evaluator version"] --> R
    R --> D["Canonical digest and diff"]
    style P fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style T fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style M fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Which record can reproduce the comparison: the new prompt string alone, or the prompt plus tool, retrieval, model, and evaluator pins? The next cell checks the missing fields rather than answering by assertion.

In [ ]:
# -- Expose the incomplete unversioned-prompt record ------------------------
releases_by_id = {release["prompt_release_id"]: release for release in release_document["releases"]}
baseline = releases_by_id["prompt-riv-001"]
candidate = releases_by_id["prompt-riv-002"]
behavior_fields = [
    "prompt_template",
    "tool_schema_version",
    "retrieval_config_id",
    "model_alias",
    "evaluator_version",
]
unversioned_record = {"prompt_template": candidate["prompt_template"]}
missing_fields = [field for field in behavior_fields if field not in unversioned_record]
assert missing_fields == ["tool_schema_version", "retrieval_config_id", "model_alias", "evaluator_version"]
print(f"Unversioned record is missing {len(missing_fields)} behavior pins: {missing_fields}")
print("Prediction resolved: the prompt string alone cannot reproduce application behavior.")

In [ ]:
# -- Diff and fingerprint the two complete release bundles ------------------
def canonical_digest(value: object) -> str:
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=True)
    return sha256(payload.encode("utf-8")).hexdigest()

def release_bundle(release: dict) -> dict:
    keys = ["prompt_release_id", *behavior_fields, "rollback_target"]
    return {key: release[key] for key in keys}

actual_changed_fields = [field for field in behavior_fields if baseline[field] != candidate[field]]
assert actual_changed_fields == candidate["changed_fields"] == ["prompt_template"]
assert candidate["rollback_target"] == baseline["prompt_release_id"]

bundle_digests = {
    baseline["prompt_release_id"]: canonical_digest(release_bundle(baseline)),
    candidate["prompt_release_id"]: canonical_digest(release_bundle(candidate)),
}
assert len(set(bundle_digests.values())) == 2
print(f"Changed behavior fields: {actual_changed_fields}")
for release_id, digest in bundle_digests.items():
    print(f"{release_id}: {digest[:16]}...")
print("TAKEAWAY: one prompt change produces a new immutable bundle while every control pin remains explicit.")

### Release-Bundle Pitfalls

| | Pattern | Why it matters |
|---|---|---|
| Wrong | `prompt = "latest"` | The observed behavior cannot be reconstructed or rolled back |
| Right | Immutable ID plus canonical digest | A trace can name the exact bundle |
| Wrong | Change prompt and evaluator in one candidate | A score change has two plausible causes |
| Right | Declare and verify `changed_fields` | The comparison isolates the intended variable |
| Wrong | Let a model alias resolve differently during the run | Paired cases no longer hold generation configuration fixed |
| Right | Resolve aliases to deployment metadata in production evidence | The logical contract and physical target are both auditable |

**Quick Health Check:** IDs are unique; declared changes equal measured changes; all behavior pins are present; candidate digest differs; rollback target exists and is accepted.

In [ ]:
# -- Run release-bundle health checks ---------------------------------------
release_ids = [release["prompt_release_id"] for release in release_document["releases"]]
rollback_target = releases_by_id.get(candidate["rollback_target"])
bundle_checks = {
    "unique_release_ids": len(release_ids) == len(set(release_ids)),
    "all_behavior_pins_present": all(candidate.get(field) for field in behavior_fields),
    "declared_diff_matches_actual": candidate["changed_fields"] == actual_changed_fields,
    "digests_differ": bundle_digests[baseline["prompt_release_id"]] != bundle_digests[candidate["prompt_release_id"]],
    "rollback_target_is_accepted": rollback_target is not None and rollback_target["status"] == "accepted",
}
assert all(bundle_checks.values()), bundle_checks
for name, passed in bundle_checks.items():
    print(f"PASS: {name} = {passed}")
print("CHECKPOINT: Riverside can now identify and reproduce both release bundles.")

**Your turn:** Change one release component. Set `EXERCISE_FIELD` to another behavior field. The exercise edits an in-memory copy only and verifies that the digest and measured diff move together.

In [ ]:
# -- Exercise: change one in-memory release component -----------------------
EXERCISE_FIELD = "model_alias"  # CHANGE THIS: choose a field from behavior_fields
assert EXERCISE_FIELD in behavior_fields
exercise_release = deepcopy(baseline)
exercise_release[EXERCISE_FIELD] = exercise_release[EXERCISE_FIELD] + "-exercise"
exercise_changed = [field for field in behavior_fields if baseline[field] != exercise_release[field]]
exercise_digest_changed = canonical_digest(release_bundle(baseline)) != canonical_digest(release_bundle(exercise_release))
assert exercise_changed == [EXERCISE_FIELD] and exercise_digest_changed
print(f"PASS: changing {EXERCISE_FIELD!r} produced exactly one measured diff and a new digest.")
print("TAKEAWAY: every behavior-affecting component participates in release identity.")

**Reflection bridge:** The complete bundle isolates the prompt change, but release identity says nothing about whether the change is safe. Compare the same cases by aggregate and critical slice next.

---

## 2 - Aggregate Improvement Can Hide a Slice Regression

Riverside's question for this section: does a 20 percentage-point aggregate gain authorize promotion when one policy-critical slice gets worse?

```mermaid
flowchart TD
    C["Five paired cases"] --> A["Aggregate gate\n60% to 80%"]
    C --> S["Slice gates\nrights finance security hr editorial"]
    A --> P["Looks promotable"]
    S --> F["Security 100% to 0%"]
    P --> G{"All gates pass?"}
    F --> G
    G -->|No| R["Reject candidate"]
    style C fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style A fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** If Riverside checks only aggregate pass rate, choose the likely verdict: promote, reject, or inconclusive. The next cell applies exactly that incomplete policy.

In [ ]:
# -- Apply the deliberately incomplete aggregate-only gate ------------------
def pass_summary(rows: list[dict], pass_field: str) -> dict:
    passed = sum(bool(row[pass_field]) for row in rows)
    total = len(rows)
    return {"passed": passed, "total": total, "pass_rate": passed / total}

aggregate = {
    "baseline": pass_summary(paired_cases, "baseline_pass"),
    "candidate": pass_summary(paired_cases, "candidate_pass"),
}
aggregate_delta = aggregate["candidate"]["pass_rate"] - aggregate["baseline"]["pass_rate"]
aggregate_only_verdict = "promote" if aggregate_delta >= 0 else "reject"
assert aggregate["baseline"]["passed"] == 3
assert aggregate["candidate"]["passed"] == 4
assert round(aggregate_delta, 10) == 0.2
print(f"Baseline: {aggregate['baseline']['passed']}/5 = {aggregate['baseline']['pass_rate']:.0%}")
print(f"Candidate: {aggregate['candidate']['passed']}/5 = {aggregate['candidate']['pass_rate']:.0%}")
print(f"Aggregate-only verdict: {aggregate_only_verdict}")
print("Prediction resolved: an aggregate-only policy would promote the candidate.")

In [ ]:
# -- Put aggregate and slice pass rates side by side ------------------------
rows_by_slice = defaultdict(list)
for case in paired_cases:
    rows_by_slice[case["slice"]].append(case)

slice_stats = {}
for slice_name, rows in sorted(rows_by_slice.items()):
    slice_stats[slice_name] = {
        "baseline": pass_summary(rows, "baseline_pass"),
        "candidate": pass_summary(rows, "candidate_pass"),
    }

labels = ["aggregate", *slice_stats.keys()]
baseline_rates = [aggregate["baseline"]["pass_rate"], *[slice_stats[name]["baseline"]["pass_rate"] for name in slice_stats]]
candidate_rates = [aggregate["candidate"]["pass_rate"], *[slice_stats[name]["candidate"]["pass_rate"] for name in slice_stats]]
positions = list(range(len(labels)))

plt.style.use("dark_background")
fig, ax = plt.subplots(figsize=(11, 4), facecolor="#1a1a2e")
ax.set_facecolor("#1a1a2e")
ax.bar([position - 0.18 for position in positions], baseline_rates, width=0.36, label="baseline", color="#1d4ed8")
ax.bar([position + 0.18 for position in positions], candidate_rates, width=0.36, label="candidate", color="#b45309")
ax.set_xticks(positions, labels)
ax.set_ylim(0, 1.08)
ax.set_ylabel("Pass rate")
ax.set_title("Riverside paired evaluation: aggregate and slices")
ax.legend()
plt.tight_layout()
plt.show()
print("TAKEAWAY: side-by-side slices reveal which users or risks pay for an aggregate gain.")

The aggregate was not false. It was incomplete. Averages allow improvements in finance and editorial cases to compensate numerically for a security failure. Riverside's release policy does not allow that trade. Safety- and security-critical gates are **non-compensating constraints**: a gain elsewhere cannot buy permission to regress them.

In [ ]:
# -- Apply Riverside's aggregate, critical-slice, and rollback gates --------
CRITICAL_SLICES = {"security", "safety"}
critical_regressions = []
for slice_name, stats in slice_stats.items():
    baseline_rate = stats["baseline"]["pass_rate"]
    candidate_rate = stats["candidate"]["pass_rate"]
    if slice_name in CRITICAL_SLICES and candidate_rate < baseline_rate:
        critical_regressions.append({
            "slice": slice_name,
            "baseline_rate": baseline_rate,
            "candidate_rate": candidate_rate,
        })

release_policy = {
    "aggregate_non_decrease": aggregate_delta >= 0,
    "no_critical_slice_regression": not critical_regressions,
    "valid_rollback_target": bundle_checks["rollback_target_is_accepted"],
}
offline_gate_pass = all(release_policy.values())
assert critical_regressions == [{"slice": "security", "baseline_rate": 1.0, "candidate_rate": 0.0}]
assert not offline_gate_pass
for gate, passed in release_policy.items():
    print(f"{'PASS' if passed else 'FAIL'}: {gate}")
print("DECISION: reject prompt-riv-002; the security slice regressed from 100% to 0%.")

### Aggregate-and-Slice Pitfalls

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Gate only the global mean | Large or easy slices can hide a rare critical failure |
| Right | Predeclare critical slices and non-compensating thresholds | The release policy exists before results are visible |
| Wrong | Create a new slice after seeing one bad row | Post-hoc slicing can manufacture a narrative |
| Right | Version slice definitions with the evaluator | Repeated release decisions remain comparable |
| Wrong | Treat one security pass as a stable 100% estimate | One case gives almost no population precision |
| Right | Block the known regression while expanding critical coverage | Policy and uncertainty answer different questions |

**Quick Health Check:** aggregate arithmetic matches row labels; every case belongs to one declared slice; critical slices are named before the decision; regressions cannot be compensated; rollback validity is a separate gate.

In [ ]:
# -- Run aggregate-and-slice health checks ---------------------------------
slice_case_count = sum(stats["baseline"]["total"] for stats in slice_stats.values())
slice_checks = {
    "aggregate_counts_match_rows": aggregate["baseline"]["total"] == len(paired_cases),
    "each_case_counted_once": slice_case_count == len(paired_cases),
    "security_is_predeclared_critical": "security" in CRITICAL_SLICES,
    "critical_regression_detected": [item["slice"] for item in critical_regressions] == ["security"],
    "offline_gate_blocks_candidate": offline_gate_pass is False,
}
assert all(slice_checks.values()), slice_checks
for name, passed in slice_checks.items():
    print(f"PASS: {name} = {passed}")
print("CHECKPOINT: the aggregate gain is retained as evidence, but it no longer controls promotion by itself.")

**Reflection bridge:** Slice gates expose the security regression, but five paired cases still cannot support a broad statistical claim. Preserve the case pairing and show exactly how weak the uncertainty evidence is.

---

## 3 - Pair the Offline Comparison and Show Its Uncertainty

Riverside's question for this section: which exact cases changed, and how much confidence can five paired observations support?

For each case, compute $d_i=c_i-b_i$. A value of `1` is a candidate win, `-1` is a candidate loss, and `0` is a tie. Pairing holds the query, reference requirements, tool schema, retrieval config, model alias, and evaluator fixed. It removes case-mix noise from the comparison; it does not make five cases representative.

```mermaid
flowchart LR
    Q["Same case i"] --> B["Baseline pass b_i"]
    Q --> C["Candidate pass c_i"]
    B --> D["Paired delta d_i"]
    C --> D
    D --> W["Wins losses ties"]
    D --> U["Exact and bootstrap\nuncertainty"]
    U --> L["Underpowered label"]
    style Q fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style W fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style U fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style L fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Classify every paired outcome ------------------------------------------
paired_outcomes = []
for case in paired_cases:
    delta = int(case["candidate_pass"]) - int(case["baseline_pass"])
    label = {1: "candidate_win", 0: "tie", -1: "candidate_loss"}[delta]
    paired_outcomes.append({
        "eval_case_id": case["eval_case_id"],
        "slice": case["slice"],
        "delta": delta,
        "outcome": label,
    })

outcome_counts = Counter(item["outcome"] for item in paired_outcomes)
assert outcome_counts == Counter({"candidate_win": 2, "tie": 2, "candidate_loss": 1})
for item in paired_outcomes:
    print(f"{item['eval_case_id']} [{item['slice']}]: {item['outcome']} (d={item['delta']:+d})")
print("TAKEAWAY: the 20-point aggregate gain is two wins, one loss, and two ties.")

**Predict:** With only three discordant pairs and a 2-to-1 split, will a two-sided exact sign test produce strong evidence against equal win/loss probability? Choose `p < 0.05`, `p around 0.25`, or `p = 1.0`.

The sign test ignores ties and asks whether wins and losses among discordant pairs are compatible with a 50/50 null. It is exact for this tiny discrete sample, but low power is the point of the demonstration.

In [ ]:
# -- Compute a two-sided exact sign test on discordant pairs ----------------
wins = outcome_counts["candidate_win"]
losses = outcome_counts["candidate_loss"]
discordant_pairs = wins + losses
more_extreme_side = max(wins, losses)
one_sided_tail = sum(comb(discordant_pairs, k) for k in range(more_extreme_side, discordant_pairs + 1)) / (2 ** discordant_pairs)
sign_test_p = min(1.0, 2 * one_sided_tail)
assert discordant_pairs == 3 and sign_test_p == 1.0
print(f"Discordant pairs: {discordant_pairs} ({wins} wins, {losses} loss)")
print(f"Two-sided exact sign-test p-value: {sign_test_p:.3f}")
print("Prediction resolved: p = 1.0; the discrete sample is far too small for a significance claim.")

In [ ]:
# -- Bootstrap the paired mean delta without breaking the pairs -------------
def percentile(sorted_values: list[float], q: float) -> float:
    index = round((len(sorted_values) - 1) * q)
    return sorted_values[index]

paired_deltas = [item["delta"] for item in paired_outcomes]
bootstrap_rng = random.Random(RANDOM_SEED)
bootstrap_draws = 10_000
bootstrap_means = []
for _ in range(bootstrap_draws):
    resampled = [bootstrap_rng.choice(paired_deltas) for _ in paired_deltas]
    bootstrap_means.append(sum(resampled) / len(resampled))
bootstrap_means.sort()
paired_interval_95 = (percentile(bootstrap_means, 0.025), percentile(bootstrap_means, 0.975))
paired_mean_delta = sum(paired_deltas) / len(paired_deltas)

print(f"Observed paired mean delta: {paired_mean_delta:+.1%}")
print(f"Illustrative paired bootstrap 95% interval: [{paired_interval_95[0]:+.1%}, {paired_interval_95[1]:+.1%}]")
print("LIMIT: five mechanism cases are underpowered; this interval is descriptive, not production evidence.")

### Paired-Comparison Pitfalls

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Compare baseline on last month's cases with candidate on this month's cases | Traffic and case mix are confounded with release |
| Right | Score both releases on the same case IDs and evaluator | Each delta has one controlled unit |
| Wrong | Bootstrap baseline and candidate rows independently | The resample destroys pairing |
| Right | Resample paired deltas or paired case records | Case difficulty remains controlled |
| Wrong | Interpret `p > 0.05` as proof of no effect | An underpowered test cannot establish equivalence |
| Right | Report effect, interval, sample size, and power limitation | The decision states what remains unknown |

**Quick Health Check:** every ID appears once per release; fixed components match; paired deltas reproduce aggregate change; ties are excluded only from the sign test; uncertainty is labeled underpowered.

In [ ]:
# -- Run paired-comparison health checks ------------------------------------
fixed_fields = ["tool_schema_version", "retrieval_config_id", "model_alias", "evaluator_version"]
paired_checks = {
    "all_case_ids_unique": len(case_ids) == len(set(case_ids)),
    "fixed_release_components_match": all(baseline[field] == candidate[field] for field in fixed_fields),
    "paired_delta_matches_aggregate": abs(paired_mean_delta - aggregate_delta) < 1e-12,
    "discordant_count_is_three": discordant_pairs == 3,
    "no_significance_claim": sign_test_p >= 0.05,
}
assert all(paired_checks.values()), paired_checks
for name, passed in paired_checks.items():
    print(f"PASS: {name} = {passed}")
print("CHECKPOINT: pairing localizes changes; uncertainty prevents overclaiming them.")

**Your turn:** Test the interval's Monte Carlo stability. Change only the number of resamples. The observed five deltas do not change, so a narrower-looking interval from more draws would reduce simulation noise, not create more evidence.

In [ ]:
# -- Exercise: change bootstrap draw count, not the evidence ----------------
EXERCISE_DRAWS = 2_000  # CHANGE THIS: try 200, 2_000, or 20_000
assert EXERCISE_DRAWS > 0
exercise_rng = random.Random(RANDOM_SEED)
exercise_means = sorted(
    sum(exercise_rng.choice(paired_deltas) for _ in paired_deltas) / len(paired_deltas)
    for _ in range(EXERCISE_DRAWS)
)
exercise_interval = (percentile(exercise_means, 0.025), percentile(exercise_means, 0.975))
assert paired_mean_delta == 0.2
print(f"Draws: {EXERCISE_DRAWS:,}; interval: [{exercise_interval[0]:+.1%}, {exercise_interval[1]:+.1%}]")
print("TAKEAWAY: more bootstrap draws stabilize computation; only more representative cases strengthen evidence.")

**Reflection bridge:** Paired offline evidence localizes the regression but cannot observe candidate behavior on realistic traffic. Shadow replay can add realism without exposure; later A/B assignment needs a stable unit.

---

## 4 - Separate Shadow Replay from A/B Assignment

Riverside's question for this section: how do you observe candidate behavior on realistic traffic without serving unsafe output, and how would you assign a later live experiment without traffic confounding?

Shadow and A/B answer different questions. Shadow mirrors the same input to both releases and returns only the baseline response. A/B assigns an exposure unit to exactly one serving release and compares outcomes across randomized populations. Shadow is safer but cannot measure user response to candidate output.

```mermaid
flowchart TD
    R["Incoming Riverside request"] --> B["Accepted baseline"]
    R -. mirrored .-> S["Candidate shadow"]
    B --> U["Response returned to user"]
    S --> O["Offline comparison only"]
    R --> H{"Later A/B hash by\nexposure unit"}
    H -->|control| B2["Baseline served"]
    H -->|treatment| C2["Candidate served"]
    style R fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style U fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style O fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B2 fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C2 fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Build a zero-user-impact shadow replay from recorded responses ---------
shadow_records = []
for case in paired_cases:
    shadow_records.append({
        "eval_case_id": case["eval_case_id"],
        "returned_release_id": baseline["prompt_release_id"],
        "returned_response": case["baseline_response"],
        "shadow_release_id": candidate["prompt_release_id"],
        "shadow_response": case["candidate_response"],
        "shadow_side_effects": "blocked",
        "shadow_pass": case["candidate_pass"],
    })

assert all(record["returned_release_id"] == "prompt-riv-001" for record in shadow_records)
assert all(record["shadow_side_effects"] == "blocked" for record in shadow_records)
shadow_failures = [record["eval_case_id"] for record in shadow_records if not record["shadow_pass"]]
print(f"Shadowed cases: {len(shadow_records)}; candidate failures: {shadow_failures}")
print("TAKEAWAY: shadow evidence observes the candidate while every user still receives the accepted release.")

**Predict:** Which hash input keeps assignment stable when release metadata changes: `(experiment_id, exposure_key)` or `(experiment_id, exposure_key, release_id)`? Including the release ID feels specific, but it can reshuffle users when the candidate changes.

In [ ]:
# -- Assign A/B variants by stable exposure unit ----------------------------
def assignment_bucket(experiment_id: str, exposure_key: str) -> float:
    digest = blake2b(f"{experiment_id}:{exposure_key}".encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(digest, "big") / (2 ** 64)

def assign_variant(experiment_id: str, exposure_key: str, candidate_fraction: float) -> str:
    if not 0.0 <= candidate_fraction <= 1.0:
        raise ValueError("candidate_fraction must be between 0 and 1")
    return "candidate" if assignment_bucket(experiment_id, exposure_key) < candidate_fraction else "baseline"

EXPERIMENT_ID = "exp-riv-prompt-002"
exposure_keys = [f"riverside-editor-{index:03d}" for index in range(1, 101)]
assignments_first = {key: assign_variant(EXPERIMENT_ID, key, 0.10) for key in exposure_keys}
assignments_second = {key: assign_variant(EXPERIMENT_ID, key, 0.10) for key in reversed(exposure_keys)}
assert assignments_first == assignments_second
candidate_count = sum(variant == "candidate" for variant in assignments_first.values())
print(f"Stable assignments: {len(assignments_first)}; candidate units at w=10%: {candidate_count}")
print("Prediction resolved: hash experiment ID plus exposure key; release IDs belong in exposure logs, not assignment input.")

In [ ]:
# -- Expose request-level contamination -------------------------------------
editor_key = "riverside-editor-017"
stable_session_assignments = [assign_variant(EXPERIMENT_ID, editor_key, 0.50) for _ in range(20)]
request_level_assignments = [
    assign_variant(EXPERIMENT_ID, f"{editor_key}:request-{index}", 0.50)
    for index in range(20)
]
stable_variants = set(stable_session_assignments)
request_variants = set(request_level_assignments)
assert len(stable_variants) == 1
print(f"Exposure-level variants across 20 requests: {sorted(stable_variants)}")
print(f"Request-level variants across 20 requests: {sorted(request_variants)}")
print("TAKEAWAY: assign at the unit that can carry treatment effects, such as user, account, or conversation.")

### Shadow-and-A/B Pitfalls

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Let shadow tool calls reach production side effects | The candidate is serving risk even if its text is discarded |
| Right | Block writes or use a faithful sandbox | Mirrored traffic remains observational |
| Wrong | Assign every request independently | One user's behavior can be influenced by both variants |
| Right | Assign by a stable exposure unit | Treatment remains consistent for the analysis unit |
| Wrong | Put release ID in the assignment hash | A release edit can reshuffle the population |
| Right | Hash experiment ID and exposure key; log the served release separately | Assignment and lineage have distinct jobs |
| Wrong | Peek daily and stop when the candidate looks good | Repeated testing inflates false-positive risk |
| Right | Pre-register sample size, duration, metrics, guardrails, and stopping rule | The online decision remains interpretable |

**Quick Health Check:** shadow output is never returned; side effects are blocked; assignments are stable and release-independent; exposure unit matches spillover risk; analysis and stop rules are declared before launch.

In [ ]:
# -- Run shadow-and-assignment health checks -------------------------------
online_checks = {
    "baseline_only_returned": all(record["returned_release_id"] == baseline["prompt_release_id"] for record in shadow_records),
    "shadow_side_effects_blocked": all(record["shadow_side_effects"] == "blocked" for record in shadow_records),
    "assignment_order_independent": assignments_first == assignments_second,
    "one_variant_per_exposure_unit": len(stable_variants) == 1,
    "release_id_not_in_hash_contract": candidate["prompt_release_id"] not in f"{EXPERIMENT_ID}:{editor_key}",
}
assert all(online_checks.values()), online_checks
for name, passed in online_checks.items():
    print(f"PASS: {name} = {passed}")
print("CHECKPOINT: Riverside can collect realistic candidate evidence without silently changing who sees which release.")

**Your turn:** Inspect a candidate traffic share. Change `EXERCISE_CANDIDATE_FRACTION`. A small finite cohort will not match the requested fraction exactly; assignment stability matters before approximate balance.

In [ ]:
# -- Exercise: change the planned A/B allocation ----------------------------
EXERCISE_CANDIDATE_FRACTION = 0.20  # CHANGE THIS: choose a value from 0.0 to 1.0
exercise_assignments = {
    key: assign_variant(EXPERIMENT_ID, key, EXERCISE_CANDIDATE_FRACTION)
    for key in exposure_keys
}
observed_fraction = sum(value == "candidate" for value in exercise_assignments.values()) / len(exercise_assignments)
assert exercise_assignments == {key: assign_variant(EXPERIMENT_ID, key, EXERCISE_CANDIDATE_FRACTION) for key in exposure_keys}
print(f"Requested candidate fraction: {EXERCISE_CANDIDATE_FRACTION:.0%}")
print(f"Observed finite-cohort fraction: {observed_fraction:.0%}")
print("TAKEAWAY: hashing gives deterministic assignment; larger randomized cohorts support balance, not exact quotas.")

**Reflection bridge:** Shadow and stable assignment define how evidence could be collected, but a known offline security failure already blocks entry. Canary must follow gates, not replace them.

---

## 5 - Require Canary Entry, Stop, and Rollback Evidence

Riverside's question for this section: what must be true before the candidate serves even 1% of traffic, and what exact action follows a failed gate?

A canary is not the first test. It follows schema checks, offline evaluation, and usually shadow observation. Entry evidence names the immutable release and rollback target. Exit evidence names the metrics, exposure window, slice results, and decision. Stop evidence names the threshold that ended exposure.

```mermaid
flowchart LR
    V["Versioned candidate"] --> O{"Offline gates"}
    O -->|pass| S["Shadow"]
    O -->|fail| R["Retain baseline"]
    S --> C{"Canary entry"}
    C -->|pass| W["Low-percent exposure"]
    W --> X{"Exit or stop gates"}
    X -->|pass| P["Ramp"]
    X -->|fail| B["Set candidate weight to zero"]
    style V fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style O fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style W fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Build canary evidence and stop before unsafe exposure ------------------
canary_evidence = {
    "candidate_release_id": candidate["prompt_release_id"],
    "candidate_bundle_digest": bundle_digests[candidate["prompt_release_id"]],
    "rollback_target": candidate["rollback_target"],
    "entry_gates": {
        "schema_valid": not release_errors and not case_errors,
        "bundle_diff_isolated": actual_changed_fields == ["prompt_template"],
        "offline_release_policy_passed": offline_gate_pass,
        "shadow_side_effects_blocked": online_checks["shadow_side_effects_blocked"],
    },
    "planned_candidate_fraction": 0.01,
    "planned_health_metrics": [
        "critical_slice_pass_rate",
        "overall_pass_rate",
        "error_rate",
        "p95_latency_ms",
        "cost_per_success",
    ],
    "stop_conditions": [
        "any confirmed safety or security regression",
        "error or latency budget exceeded",
        "release lineage missing from an exposure record",
    ],
}
canary_evidence["entry_allowed"] = all(canary_evidence["entry_gates"].values())
canary_evidence["status"] = "eligible" if canary_evidence["entry_allowed"] else "blocked_offline"
assert canary_evidence["status"] == "blocked_offline"
print(json.dumps(canary_evidence, indent=2))
print("DECISION: do not expose 1% of users; the offline security gate already failed.")

In [ ]:
# -- Record the correct rollback action for a pre-exposure failure -----------
rollback_evidence = {
    "trigger": "offline_critical_slice_regression",
    "candidate_release_id": candidate["prompt_release_id"],
    "active_release_before": baseline["prompt_release_id"],
    "rollback_target": candidate["rollback_target"],
    "routing_action": "retain_active_baseline",
    "candidate_traffic_after": 0.0,
    "cache_action_if_previously_exposed": "invalidate_or_rekey_by_release_id",
    "session_action_if_previously_exposed": "pin_or_restart_under_accepted_release",
    "side_effect_compensation": "not_performed_by_release_rollback",
    "candidate_record_retained_for_audit": True,
}
assert rollback_evidence["rollback_target"] == rollback_evidence["active_release_before"]
assert rollback_evidence["candidate_traffic_after"] == 0.0
print(json.dumps(rollback_evidence, indent=2))
print("TAKEAWAY: because canary never started, Riverside retains the baseline pointer instead of claiming a production rollback.")

### Canary-and-Rollback Pitfalls

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Use canary traffic to discover a known offline security failure | Real users absorb avoidable risk |
| Right | Require offline and shadow entry evidence first | Canary answers residual production questions |
| Wrong | Say `rollback to latest good` | The target is mutable and unauditable |
| Right | Name the accepted immutable release ID and digest | Routing can make one exact pointer change |
| Wrong | Delete the rejected candidate | The failed diff and evaluation disappear |
| Right | Retain candidate, report, trigger, and decision | Future reviews can reconstruct why it was blocked |
| Wrong | Treat release rollback as undo for sent emails or writes | Config rollback cannot reverse business side effects |
| Right | Use separate compensation workflows for committed actions | Deployment recovery and action recovery stay distinct |

**Quick Health Check:** all entry gates are explicit; failed offline gates force zero exposure; rollback target is accepted and immutable; candidate evidence is retained; cache/session actions are release-aware; side-effect compensation is separate.

In [ ]:
# -- Run canary-and-rollback health checks ---------------------------------
canary_checks = {
    "offline_failure_blocks_entry": not canary_evidence["entry_allowed"],
    "candidate_traffic_is_zero": rollback_evidence["candidate_traffic_after"] == 0.0,
    "rollback_target_matches_accepted_release": rollback_evidence["rollback_target"] == baseline["prompt_release_id"],
    "candidate_evidence_retained": rollback_evidence["candidate_record_retained_for_audit"],
    "side_effects_not_conflated": rollback_evidence["side_effect_compensation"] == "not_performed_by_release_rollback",
}
assert all(canary_checks.values()), canary_checks
for name, passed in canary_checks.items():
    print(f"PASS: {name} = {passed}")
print("CHECKPOINT: Riverside has evidence for why exposure stayed at zero and which release remains active.")

**Your turn:** Test whether an aggregate threshold can override a critical gate. Raise the required aggregate improvement. The final conjunction must still reject any candidate with a critical-slice regression.

In [ ]:
# -- Exercise: keep critical gates non-compensating -------------------------
EXERCISE_MIN_AGGREGATE_DELTA = 0.10  # CHANGE THIS: try -0.10, 0.00, or 0.30
exercise_aggregate_gate = aggregate_delta >= EXERCISE_MIN_AGGREGATE_DELTA
exercise_critical_gate = not critical_regressions
exercise_release_gate = exercise_aggregate_gate and exercise_critical_gate and bundle_checks["rollback_target_is_accepted"]
assert exercise_release_gate is False
print(f"Aggregate gate passed: {exercise_aggregate_gate}")
print(f"Critical-slice gate passed: {exercise_critical_gate}")
print("PASS: the combined release gate rejects the candidate regardless of aggregate threshold.")

**Reflection bridge:** The candidate never earned exposure, so the correct rollback action is to retain the accepted baseline and preserve the rejected evidence. The final step packages that reasoning for audit.

---

## 6 - Retain One Auditable Decision

Riverside's question for this section: can a reviewer reconstruct the release, comparison, uncertainty, assignment boundary, blocked canary, and rollback target from one record?

```mermaid
flowchart LR
    B["Bundle evidence"] --> R["Release decision report"]
    S["Aggregate and slice gates"] --> R
    U["Paired uncertainty"] --> R
    A["Shadow and assignment checks"] --> R
    C["Canary and rollback evidence"] --> R
    R --> D["REJECT prompt-riv-002"]
    D --> K["KEEP prompt-riv-001 active"]
    style B fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style U fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style A fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style K fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Assemble the final in-memory release decision report -------------------
fixture_hashes_after = {name: sha256(path.read_bytes()).hexdigest() for name, path in fixture_paths.items()}
release_decision_report = {
    "schema_version": "ai-eng.prompt-release-decision.v1",
    "baseline_release_id": baseline["prompt_release_id"],
    "candidate_release_id": candidate["prompt_release_id"],
    "bundle_digests": bundle_digests,
    "changed_fields": actual_changed_fields,
    "aggregate": aggregate,
    "paired_outcomes": dict(outcome_counts),
    "paired_mean_delta": paired_mean_delta,
    "exact_sign_test_p": sign_test_p,
    "paired_bootstrap_interval_95": list(paired_interval_95),
    "uncertainty_label": "underpowered_mechanism_fixture",
    "critical_regressions": critical_regressions,
    "release_policy": release_policy,
    "shadow_failures": shadow_failures,
    "canary_status": canary_evidence["status"],
    "rollback_evidence": rollback_evidence,
    "fixture_bytes_unchanged": fixture_hashes_after == fixture_hashes_before,
    "decision": "reject_candidate_retain_baseline",
    "decision_reason": "security slice regressed from pass to fail",
}
assert release_decision_report["fixture_bytes_unchanged"]
assert release_decision_report["decision"] == "reject_candidate_retain_baseline"
print(json.dumps(release_decision_report, indent=2, sort_keys=True))
print("FINAL DECISION: reject prompt-riv-002 and retain prompt-riv-001.")

### Practitioner Release Checklist

Before proposing a prompt or application-config release, retain evidence for each question:

1. **Identity:** Does one immutable ID pin prompt, tools, retrieval, model, and evaluator?
2. **Diff:** Do declared changed fields equal the measured bundle diff?
3. **Offline comparison:** Did baseline and candidate see the same cases and fixed components?
4. **Slices:** Are safety, security, tenant, language, and other critical slices predeclared?
5. **Uncertainty:** Are effect size, interval/test, sample size, and power limitations reported together?
6. **Shadow:** Are candidate outputs withheld and side effects blocked or sandboxed?
7. **A/B:** Is assignment stable at the correct exposure unit with a pre-registered stopping rule?
8. **Canary:** Do entry, stop, exit, latency, error, cost, and quality gates exist before exposure?
9. **Rollback:** Is the accepted target immutable, routable, cache-aware, session-aware, and separate from action compensation?
10. **Audit:** Can one report reconstruct the evidence and decision without a mutable dashboard?

### Honest Limits

- Five hand-labeled fixture cases prove arithmetic and control flow, not population quality.
- One security case is enough to block a known regression under Riverside policy, but not enough to estimate a stable security pass rate.
- Pre-recorded responses remove model nondeterminism; real evaluation must repeat stochastic generations or otherwise model run-to-run variation.
- Supplied pass labels do not validate the evaluator. Human agreement, rubric validity, judge calibration, and evaluator drift need separate evidence.
- Deterministic hashing demonstrates assignment, not randomization balance, sample-ratio-mismatch monitoring, novelty effects, or causal validity.
- Shadow traffic cannot measure user reaction to candidate output and must not execute real side effects.
- A/B tests need privacy review, exposure logging, pre-registered metrics, minimum detectable effect and power planning, duration rules, and sequential-testing controls.
- Local canary logic does not prove gateway routing, latency, cost, capacity, cache invalidation, or cloud rollback.
- Release rollback stops future use of a config; it does not undo already committed external actions.

### Three-Tier Coverage Ledger

| Tier | Technique | Evidence or reason |
|---|---|---|
| Built and measured | Immutable prompt release bundle | Component presence, measured diff, rollback target, and canonical digest checks |
| Built and measured | Aggregate comparison | Deterministic 3/5 versus 4/5 calculation |
| Built and measured | Critical-slice regression gate | Security changes from 1/1 pass to 0/1 pass and blocks promotion |
| Built and measured | Paired offline comparison | Two wins, one loss, two ties on stable case IDs |
| Built and measured | Exact sign test | Three discordant pairs produce an explicitly underpowered result |
| Built and measured | Paired bootstrap | Resamples paired deltas with a fixed seed and reports a descriptive interval |
| Built and measured | Shadow replay | Baseline response served; candidate response observed; side effects blocked |
| Built and measured | Stable A/B assignment | Release-independent hash by experiment and exposure unit |
| Built and measured | Canary preflight | Failed offline gate forces zero candidate exposure |
| Built and measured | Rollback evidence | Accepted target retained, candidate preserved, cache/session actions named |
| Explained and illustrated | Sequential testing and peeking control | Pitfall and pre-registration requirement are shown; no online observations exist |
| Explained and illustrated | Sample-ratio mismatch and exposure-unit contamination | Assignment examples show the mechanism; no production telemetry exists |
| Named with a reason | CUPED and covariate adjustment | Five fixture rows contain no suitable pre-experiment covariates |
| Named with a reason | Stratified randomization | Production strata and sample sizes are absent |
| Named with a reason | Multi-armed bandits | Adaptive optimization answers a different question from a fixed release comparison |
| Named with a reason | Live LLM-as-judge | Default path forbids network and LLM calls; evaluator reliability belongs in the evaluation track |
| Named with a reason | Cloud canary and rollback | Requires authenticated gateway, deployment, telemetry, load, and recovery evidence |

If you find a technique named above that does not appear in this ledger, that is exactly the accounting bug this section exists to catch.

### Completed Roadmap and Riverside Decision

| Step | What failed | What now controls it | Measured fixture result |
|---:|---|---|---|
| 0 | A plausible aggregate gain invited a rushed release | Frozen fixture contract | Five unchanged paired rows validated |
| 1 | A prompt string omitted four behavior pins | Immutable release bundle | Only `prompt_template` changed; digests differ |
| 2 | Aggregate 60% to 80% hid risk | Non-compensating slice gate | Security 100% to 0%; gate fails |
| 3 | Five cases looked conclusive | Paired outcomes and uncertainty | 2 wins, 1 loss, 2 ties; exact test underpowered |
| 4 | Online traffic could be confounded | Shadow plus stable exposure assignment | Baseline only served; assignment stable |
| 5 | Canary could become the first safety test | Entry and rollback evidence | Canary blocked; candidate traffic remains 0% |
| 6 | Evidence could disappear into prose | One decision report | Reject `prompt-riv-002`; retain `prompt-riv-001` |

**Key takeaways**

- A prompt release is the prompt plus every configuration that can change behavior or its measurement.
- Aggregate improvement never compensates for a predeclared critical-slice regression.
- Pairing controls case difficulty; it does not create representativeness or power.
- Shadow observes candidate behavior without serving it; A/B measures outcomes after stable randomized exposure.
- Canary is residual-risk evidence, not permission to skip a failed offline gate.
- Rollback evidence names an immutable target and future routing action; committed side effects need compensation.

> **Next:** [Release Registry and Lineage](../04-release-registry-and-lineage/release-registry-and-lineage.ipynb) will place this prompt decision inside a wider application release that also pins model, adapter, tokenizer, index, deployment, source commit, and evaluator artifacts.